# A股市场广度 / 阿姆氏指标(TRIN) / 中金所会员持仓 数据拉取

输出 `a_breadth.csv` 和 `cffex_holding.csv`，供 `charts.ipynb` 出图。中途断了重新运行「拉取」那一格即可，会跳过已拉过的日期。

在 GitHub 上运行时 token 从仓库 Secret `TUSHARE_TOKEN` 读取。

**TRIN（Arms Index，标准公式）= (上涨家数 / 下跌家数) / (上涨成交量 / 下跌成交量)**，平盘股不计入。`trin` 为标准口径（成交量）；`trin_amt` 为按成交额的非标准变体，仅作对照。`*_ma10` 为 10 日均线（Arms 原始阈值：<0.8 超买，>1.2 超卖），`*_sum10` 为 10 日累计口径。前缀 `big_` 为当日市值前 300，`small_` 为其余。

接口规则（已按 tushare 文档核对）：
- `daily`：vol 单位手、amount 单位千元，单次上限 6000 行 → offset 翻页
- `daily_basic`：total_mv 单位万元，单次上限 6000 行；`limit_status` 需在 fields 显式请求（2/3 涨停，5/6 跌停）
- `stk_limit`：含基金，单次上限 5800 行 → 仅作兜底且翻页
- `fut_holding`：需 2000 积分 → 自动探测调用方式，只采用返回结果基本都是中金所品种的方式
- `opt_basic`（5000 积分）取合约的标的和认购/认沽；`opt_daily`（2000 积分，单次上限 15000 行）取成交量和持仓量，算 PCR
- 6 线程并发，全局限速 450 次/分钟（10000 积分上限 500 次/分钟）

In [1]:
# 首次运行先装依赖：
# !pip install tushare pandas


## 拉数据

全部步骤都在下面这一格里，运行它即可。中途断了重新运行，会跳过已拉过的日期。

In [2]:
# ===================== 拉数据（一整格，可重复运行，自动跳过已拉过的日期） =====================
import os, re, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import tushare as ts

# ---------- 配置 ----------
TOKEN = os.environ.get("TUSHARE_TOKEN", "")   # GitHub 上从 Secrets 读取
START = "20240101"
END = time.strftime("%Y%m%d")
BIG_N = 300            # 当日总市值前 300 视为"权重"，其余为"中小盘"
FUT_ENABLE = True      # 中金所会员持仓（回测显示对涨跌无预测力；嫌慢可改 False）
OPT_ENABLE = True      # ETF 期权 PCR
WORKERS = 6            # 并发线程数
MAX_PER_MIN = 450      # 10000 积分上限 500 次/分钟，留余量
os.makedirs("data", exist_ok=True)
OUT_BREADTH, OUT_FUT, OUT_OPT = "data/a_breadth.csv", "data/cffex_holding.csv", "data/option_pcr.csv"
KEY_DATES = ["20260323", "20260721", "20260921", "20260922", "20261022", "20261023"]

pro = ts.pro_api(TOKEN)
pro._DataApi__token = TOKEN
pro._DataApi__http_url = os.environ.get("TUSHARE_HTTP_URL", "http://jiaoch.site")
assert TOKEN, "没有 token：请在 GitHub 仓库 Settings → Secrets → Actions 里添加 TUSHARE_TOKEN"

# ---------- 调用：全局限速 + 重试 ----------
_lock, _stamps = threading.Lock(), []


def _throttle():
    while True:
        with _lock:
            now = time.time()
            while _stamps and now - _stamps[0] > 60:
                _stamps.pop(0)
            if len(_stamps) < MAX_PER_MIN:
                _stamps.append(now)
                return
            wait = 60 - (now - _stamps[0]) + 0.05
        time.sleep(wait)


def call(fn, **kw):
    for i in range(5):
        _throttle()
        try:
            df = getattr(pro, fn)(**kw)
            return df if df is not None else pd.DataFrame()
        except Exception as e:
            msg = str(e)
            if i == 4:
                print(f"  {fn} {kw.get('trade_date', '')} 放弃：{msg[:100]}")
            time.sleep(3 * (i + 1) if ("每分钟" in msg or "频" in msg) else 1.5 * (i + 1))
    return pd.DataFrame()


def call_all(fn, page, **kw):
    """按 offset 翻页；page 取接口单次上限，正常一次拿完。"""
    out, off = [], 0
    while True:
        df = call(fn, limit=page, offset=off, **kw)
        if df.empty:
            break
        out.append(df)
        if len(df) < page:
            break
        off += page
    return pd.concat(out, ignore_index=True).drop_duplicates() if out else pd.DataFrame()


def load(path):
    return pd.read_csv(path, dtype={"trade_date": str}) if os.path.exists(path) else pd.DataFrame()

# ---------- 市场广度 / TRIN / 大小盘拆分 ----------


def trin(n_up, n_dn, a_up, a_dn):
    if n_dn == 0 or a_up == 0 or a_dn == 0:
        return None
    return (n_up / n_dn) / (a_up / a_dn)


def group_stats(df, prefix):
    up, dn = df.pct_chg > 0, df.pct_chg < 0
    n_up, n_dn = int(up.sum()), int(dn.sum())
    a_up, a_dn = df.loc[up, "amount"].sum() / 1e5, df.loc[dn, "amount"].sum() / 1e5  # 千元 -> 亿元
    tot = df["amount"].sum() / 1e5
    v_up, v_dn = df.loc[up, "vol"].sum() / 1e6, df.loc[dn, "vol"].sum() / 1e6          # 手 -> 百万手
    return {
        f"{prefix}vol": round(df["vol"].sum() / 1e6, 3),
        f"{prefix}up_vol": round(v_up, 3), f"{prefix}dn_vol": round(v_dn, 3),
        f"{prefix}trin": trin(n_up, n_dn, v_up, v_dn),          # 标准 Arms Index：按成交量，平盘不计
        f"{prefix}n": len(df), f"{prefix}n_up": n_up, f"{prefix}n_dn": n_dn,
        f"{prefix}amt": round(tot, 2), f"{prefix}up_amt": round(a_up, 2), f"{prefix}dn_amt": round(a_dn, 2),
        f"{prefix}up_amt_share": round(a_up / tot, 4) if tot else None,
        f"{prefix}trin_amt": trin(n_up, n_dn, a_up, a_dn),      # 成交额口径（博主用的），仅作对照
    }


def day_breadth(d):
    px = call_all("daily", 6000, trade_date=d)
    if px.empty:
        return None
    px = px[px.ts_code.str.endswith((".SH", ".SZ"))].dropna(subset=["pct_chg"])
    mv = call_all("daily_basic", 6000, trade_date=d, fields="ts_code,total_mv,limit_status")
    if mv.empty or len(mv) < 0.9 * len(px):   # 每日指标 15–17 点才更新完；没齐就整天跳过，下次运行再拉
        return None
    df = px.merge(mv, on="ts_code", how="left")
    row = {"trade_date": d}
    row.update(group_stats(df, ""))
    row["n_flat"] = row["n"] - row["n_up"] - row["n_dn"]
    row["breadth_up"] = round(row["n_up"] / row["n"], 4)
    ls = pd.to_numeric(df["limit_status"], errors="coerce") if "limit_status" in df else None
    if ls is not None and ls.notna().any():
        row["limit_up"], row["limit_dn"] = int(ls.isin([2, 3]).sum()), int(ls.isin([5, 6]).sum())
    big = set(df.dropna(subset=["total_mv"]).nlargest(BIG_N, "total_mv").ts_code)
    row.update(group_stats(df[df.ts_code.isin(big)], "big_"))
    row.update(group_stats(df[~df.ts_code.isin(big)], "small_"))
    return row

# ---------- 中金所会员持仓 ----------


FUT_PRODUCTS = ["IF", "IH", "IC", "IM"]
CFFEX_PREFIX = ("IF", "IH", "IC", "IM", "IO", "HO", "MO", "T", "TF", "TS", "TL")
FUT_MODE = None


def fut_query(d, mode, prod):
    if mode == "by_product_ex":
        return call("fut_holding", trade_date=d, symbol=prod, exchange="CFFEX")
    if mode == "by_product":
        return call("fut_holding", trade_date=d, symbol=prod)
    return call("fut_holding", trade_date=d, exchange="CFFEX")


def probe_fut(dates):
    """每种方式在最近几天都试一遍；返回结果必须基本都是中金所品种才采用
    （中转站可能忽略 exchange 参数，把全国期货都返回来——那样又慢又可能被截断）。"""
    print("探测期货持仓接口：")
    recent = list(reversed(dates[-6:]))
    for mode in ["by_product_ex", "by_product", "by_exchange"]:
        for d in recent:
            df = fut_query(d, mode, "IF")
            n = len(df)
            if not n or "symbol" not in df:
                continue
            sym = df.symbol.astype(str)
            ok_share = sym.str.startswith("IF").mean() if mode != "by_exchange" else sym.str.match(r"^(IF|IH|IC|IM|IO|HO|MO|TS|TF|TL|T)\d*$").mean()
            print(f"  {d} {mode}: {n} 行，目标品种占比 {ok_share:.0%}，样例 {sorted(set(sym))[:6]}")
            if ok_share > 0.9:
                print(f"  → 使用 {mode}")
                return mode
            break   # 这种方式返回了别的品种，换下一种
    return None


def day_fut(d):
    if FUT_MODE == "by_exchange":
        raw = fut_query(d, FUT_MODE, None)
        parts = {p: raw for p in FUT_PRODUCTS}
    else:
        parts = {p: fut_query(d, FUT_MODE, p) for p in FUT_PRODUCTS}
    rows = []
    for prod, fh in parts.items():
        if fh.empty or "symbol" not in fh:
            continue
        sym = fh.symbol.astype(str)
        agg = fh[sym == prod]
        src = "product" if not agg.empty else "contracts_sum"
        fh = agg.copy() if not agg.empty else fh[sym.str.match(rf"^{prod}\d{{4}}$")].copy()
        if fh.empty:
            continue
        for c in ["long_hld", "short_hld", "vol"]:
            fh[c] = pd.to_numeric(fh[c], errors="coerce").fillna(0)
        g = fh.groupby("broker", as_index=False)[["long_hld", "short_hld", "vol"]].sum()
        g["product"], g["trade_date"], g["source"] = prod, d, src
        rows.extend(g.to_dict("records"))
    return rows

# ---------- ETF 期权 PCR ----------
# opt_basic（5000 积分）：合约信息，取标的与认购/认沽；opt_daily（2000 积分）：vol 单位手，oi 单位手，单次 15000 行


OPT_GROUPS = {"50": {"510050"}, "300": {"510300", "159919"}, "500": {"510500", "159922"},
              "kc50": {"588000", "588080"}, "cyb": {"159915"}}
NAME_HINT = [("科创", "588000"), ("创业板", "159915"), ("500", "510500"), ("300", "510300"), ("50", "510050")]
OPT_MAP = None


def underlying_of(r):
    for f in ("opt_code", "name"):
        m = re.search(r"(5[0-9]{5}|15[0-9]{4})", str(r.get(f, "")))
        if m:
            return m.group(1)
    nm = str(r.get("name", ""))
    for k, code in NAME_HINT:
        if k in nm:
            return code
    return None


def cp_of(v, name=""):
    s = f"{v}{name}".upper()
    if s.startswith("C") or "认购" in s or "购" in str(name):
        return "C"
    if s.startswith("P") or "认沽" in s or "沽" in str(name):
        return "P"
    return None


def build_opt_map():
    # 单次上限文档未写，用较小的页长翻页，避免被截断
    parts = [call_all("opt_basic", 1000, exchange=ex, fields="ts_code,name,opt_code,call_put,exchange") for ex in ("SSE", "SZSE")]
    ob = pd.concat([p for p in parts if not p.empty], ignore_index=True) if any(not p.empty for p in parts) else pd.DataFrame()
    if ob.empty:
        print("⚠ opt_basic 没有返回数据（需要 5000 积分），跳过期权 PCR")
        return None
    ob = ob[ob.ts_code.astype(str).str.endswith((".SH", ".SZ"))].drop_duplicates("ts_code")
    ob["und"] = ob.apply(underlying_of, axis=1)
    ob["cp"] = [cp_of(v, n) for v, n in zip(ob.call_put, ob.name)]
    ob = ob.dropna(subset=["und", "cp"])
    ob[["ts_code", "name", "und", "cp"]].to_csv("data/opt_basic.csv", index=False, encoding="utf-8-sig")
    print(f"期权合约 {len(ob)} 个，标的分布：{ob.und.value_counts().head(8).to_dict()}")
    return ob.set_index("ts_code")[["und", "cp"]]


def day_opt(d):
    raw = [call_all("opt_daily", 15000, trade_date=d, exchange="SSE")]
    if raw[0].empty or not raw[0].ts_code.astype(str).str.endswith(".SZ").any():   # 中转站若忽略 exchange，第一次就已含深市
        raw.append(call_all("opt_daily", 15000, trade_date=d, exchange="SZSE"))
    od = pd.concat([r for r in raw if not r.empty], ignore_index=True) if any(not r.empty for r in raw) else pd.DataFrame()
    if od.empty:
        return None
    od = od.drop_duplicates("ts_code").join(OPT_MAP, on="ts_code", how="inner")
    for c in ("vol", "oi"):
        od[c] = pd.to_numeric(od[c], errors="coerce").fillna(0)
    row = {"trade_date": d}
    groups = {"all": od, **{g: od[od.und.isin(codes)] for g, codes in OPT_GROUPS.items()}}
    for g, x in groups.items():
        cv, pv = x.loc[x.cp == "C", "vol"].sum(), x.loc[x.cp == "P", "vol"].sum()
        co, po = x.loc[x.cp == "C", "oi"].sum(), x.loc[x.cp == "P", "oi"].sum()
        row.update({f"{g}_call_vol": cv, f"{g}_put_vol": pv, f"{g}_call_oi": co, f"{g}_put_oi": po,
                    f"{g}_pcr_vol": pv / cv if cv else None, f"{g}_pcr_oi": po / co if co else None})
    return row

# ---------- 汇总 ----------


def add_smoothing(b, n=10):
    """Arms 原始用法看 10 日均线：<0.8 超买，>1.2 超卖；另算 10 日累计口径。"""
    for pre in ["", "big_", "small_"]:
        for col in ["trin", "trin_amt"]:
            if pre + col in b:
                b[f"{pre}{col}_ma{n}"] = b[pre + col].rolling(n).mean()
        r = {k: b[pre + k].rolling(n).sum() for k in ["n_up", "n_dn", "up_vol", "dn_vol"] if pre + k in b}
        if len(r) == 4:
            b[f"{pre}trin_sum{n}"] = (r["n_up"] / r["n_dn"]) / (r["up_vol"] / r["dn_vol"])
    return b


def save(path, old, rows, keys):
    allr = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
    if not allr.empty:
        allr.drop_duplicates(keys, keep="last").sort_values("trade_date").to_csv(path, index=False, encoding="utf-8-sig")


t0 = time.time()
cal = call("trade_cal", exchange="SSE", start_date=START, end_date=END, is_open="1")
dates = sorted(cal["cal_date"].astype(str))
print(f"交易日 {len(dates)} 天：{dates[0]} ~ {dates[-1]}")

if os.path.exists(OUT_BREADTH):
    have = set(pd.read_csv(OUT_BREADTH, nrows=1).columns)
    if not {"up_vol", "dn_vol", "trin_amt"} <= have:
        os.replace(OUT_BREADTH, OUT_BREADTH.replace(".csv", "_old.csv"))
        print("发现旧版数据（缺少新字段），已改名为 *_old.csv，重新拉取")

b_old, f_old, o_old = load(OUT_BREADTH), load(OUT_FUT), load(OUT_OPT)
b_done = set(b_old.trade_date) if not b_old.empty else set()
f_done = set(f_old.trade_date) if not f_old.empty else set()
o_done = set(o_old.trade_date) if not o_old.empty else set()

FUT_MODE = probe_fut(dates) if FUT_ENABLE and set(dates) - f_done else None
if FUT_ENABLE and FUT_MODE is None and set(dates) - f_done:
    print("⚠ 期货持仓没有找到只返回中金所数据的调用方式，本次跳过期货部分")
OPT_MAP = build_opt_map() if OPT_ENABLE and set(dates) - o_done else None

todo = [d for d in dates if d not in b_done or (FUT_MODE and d not in f_done) or (OPT_MAP is not None and d not in o_done)]
print(f"需要拉取 {len(todo)} 天（已有 {len(dates) - len(todo)} 天）；{WORKERS} 线程并发")


def job(d):
    r = {"d": d, "b": None, "f": [], "o": None}
    if d not in b_done:
        r["b"] = day_breadth(d)
    if FUT_MODE and d not in f_done:
        r["f"] = day_fut(d)
    if OPT_MAP is not None and d not in o_done:
        r["o"] = day_opt(d)
    return r


b_rows, f_rows, o_rows = [], [], []
with ThreadPoolExecutor(WORKERS) as ex:
    futs = [ex.submit(job, d) for d in todo]
    for i, fu in enumerate(as_completed(futs), 1):
        r = fu.result()
        if r["b"]:
            b_rows.append(r["b"])
        f_rows.extend(r["f"])
        if r["o"]:
            o_rows.append(r["o"])
        if i % 40 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  进度 {i}/{len(todo)}  已用 {el / 60:.1f} 分钟，预计还需 {el / i * (len(todo) - i) / 60:.1f} 分钟")
            save(OUT_BREADTH, b_old, b_rows, "trade_date")
            save(OUT_FUT, f_old, f_rows, ["trade_date", "product", "broker"])
            save(OUT_OPT, o_old, o_rows, "trade_date")

# 并入指数收盘价 + 平滑
b = load(OUT_BREADTH)
for code, name in [("000001.SH", "sh"), ("000300.SH", "hs300"), ("000852.SH", "zz1000")]:
    idx = call("index_daily", ts_code=code, start_date=START, end_date=END)
    if not idx.empty:
        idx = idx[["trade_date", "close", "pct_chg"]].rename(columns={"close": f"{name}_close", "pct_chg": f"{name}_pct"})
        b = b.drop(columns=[c for c in idx.columns if c != "trade_date" and c in b], errors="ignore")
        b = b.merge(idx.astype({"trade_date": str}), on="trade_date", how="left")
b = add_smoothing(b.sort_values("trade_date").reset_index(drop=True))
b.to_csv(OUT_BREADTH, index=False, encoding="utf-8-sig")

cols = ["trade_date", "breadth_up", "limit_up", "trin", "trin_amt", "trin_ma10", "amt", "big_trin", "small_trin", "sh_close"]
print("\n博主提到的几天：")
print(b[b.trade_date.isin(KEY_DATES)][[c for c in cols if c in b]].to_string(index=False))
o = load(OUT_OPT)
if not o.empty:
    print("\n期权 PCR（最近 5 天）：")
    print(o.tail(5)[["trade_date", "all_pcr_vol", "all_pcr_oi", "50_pcr_vol", "300_pcr_vol"]].round(3).to_string(index=False))
print(f"\n完成，用时 {(time.time() - t0) / 60:.1f} 分钟。数据在 data/ 目录。")


交易日 683 天：20240102 ~ 20261030
探测期货持仓接口：


  20261030 by_exchange: 724 行，目标品种占比 100%，样例 ['IC2611', 'IC2612', 'IC2703', 'IC2706', 'IF2611', 'IF2612']
  → 使用 by_exchange


期权合约 19765 个，标的分布：{'510050': 4816, '159919': 2819, '510300': 2801, '159915': 1843, '159922': 1674, '159901': 1623, '510500': 1620, '588080': 1308}
需要拉取 1 天（已有 682 天）；6 线程并发


  进度 1/1  已用 0.9 分钟，预计还需 0.0 分钟



博主提到的几天：
trade_date  breadth_up  limit_up     trin  trin_amt  trin_ma10      amt  big_trin  small_trin  sh_close
  20260323      0.0561        38 0.300710  0.317282   0.966364 24311.60  0.458090    0.293518 3813.2829
  20260721      0.5523       120 0.756303  0.290436   0.842592 29569.03  1.150184    0.722511 3864.3671
  20260921      0.8094       104 0.942318  1.840610   0.903629 20313.61  0.563026    0.937476 3949.9068
  20260922      0.4385        64 0.970866  0.864333   0.933347 21353.66  1.045783    0.937029 3952.1280
  20261022      0.0859        33 0.504959  0.509081   0.821849 15283.27  0.731707    0.532660 3658.0256
  20261023      0.8042        82 1.457127  1.066068   0.888603 19559.25  1.626581    1.339775 3690.4753

期权 PCR（最近 5 天）：
trade_date  all_pcr_vol  all_pcr_oi  50_pcr_vol  300_pcr_vol
  20261026        0.740       0.519       0.690        0.793
  20261027        0.749       0.489       0.781        0.769
  20261028        0.745       0.476       0.688        0.619
 